# V7_A_N03 — Quality Before Yield

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft; synthetic data only. Specialist and institutional approval required.

## Decision contract
Audit agricultural records before estimation, modelling, dashboards, or release. Flags request correction or documented acceptance; they do not prove falsification or poor fieldwork.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7903);n=180;df=pd.DataFrame({'record':[f'R{i:03d}' for i in range(n)],'area_ha':rng.uniform(0,45,n),'production_t':rng.uniform(0,160,n),'harvested_ha':rng.uniform(0,50,n),'unit_known':rng.choice([0,1],n,p=[.08,.92]),'source_age':rng.integers(0,20,n)});df.loc[:5,'production_t']=df.loc[:5,'area_ha']*12;df.head()

## Evidence and quality contract
Test identifiers, scope, reference time, crop/livestock codes, units, skip logic, ranges, internal consistency, duplicates, coverage, revisions, weights, and provenance.

In [2]:
df['yield_t_ha']=df.production_t/df.area_ha.clip(lower=.1);df['area_conflict']=df.harvested_ha>df.area_ha*1.1;df['yield_flag']=df.yield_t_ha>8;df['critical']=df.area_conflict|df.yield_flag|(df.unit_known==0);print(df[['area_conflict','yield_flag','critical']].sum().to_string())

area_conflict     85
yield_flag        42
critical         102


## Uncertainty, sensitivity, and abstention
Rules have false positives. Compare tolerances, route contextual exceptions to human review, and block downstream use when critical fields are unresolved.

In [3]:
df['status']=np.where(df.unit_known==0,'BLOCK—VERIFY UNIT',np.where(df.area_conflict,'VERIFY AREA CONSISTENCY',np.where(df.yield_flag,'VERIFY EXTREME YIELD','PASS TO NEXT GATE')));result=df[df.status!='PASS TO NEXT GATE'][['record','area_ha','harvested_ha','production_t','yield_t_ha','status']];print(result.head(18).round(2).to_string(index=False));print({'yield8':int((df.yield_t_ha>8).sum()),'yield10':int((df.yield_t_ha>10).sum())})

record  area_ha  harvested_ha  production_t  yield_t_ha                  status
  R000     9.47         27.73        113.65       12.00       BLOCK—VERIFY UNIT
  R001    32.25         39.98        386.96       12.00 VERIFY AREA CONSISTENCY
  R002     2.85         15.53         34.25       12.00 VERIFY AREA CONSISTENCY
  R003    10.15         43.08        121.76       12.00 VERIFY AREA CONSISTENCY
  R004    28.70          2.80        344.44       12.00    VERIFY EXTREME YIELD
  R005    39.24         17.15        470.82       12.00    VERIFY EXTREME YIELD
  R009     8.76         48.70        159.73       18.24 VERIFY AREA CONSISTENCY
  R010    25.18         38.84        127.49        5.06 VERIFY AREA CONSISTENCY
  R016    14.72         28.65         15.15        1.03 VERIFY AREA CONSISTENCY
  R018    19.74         27.64        129.16        6.54 VERIFY AREA CONSISTENCY
  R020    26.14         40.30        110.72        4.24 VERIFY AREA CONSISTENCY
  R021    30.02         40.54        154

## Decision product and authority boundary
The output is a traceable review queue with reason codes, evidence date, uncertainty, accountable owner, and status. It does not replace legal, professional, clinical, procurement, enforcement, safeguarding, budget, or ethical authority.

## Exercises
1. Add duplicate detection. 2. Explain edit versus imputation. 3. Design correction lineage.

## Exact solutions
1. Use stable record keys plus content hashes and fieldwork metadata. 2. An edit corrects with evidence; imputation estimates a missing value and must retain flags and uncertainty. 3. Preserve original, corrected value, rule, reason, editor, approver, timestamp, and version.

In [4]:
assert len(result)>0 and result['status'].notna().all();print('V7_A_N03_COMPLETE_EXECUTION_PASS')

V7_A_N03_COMPLETE_EXECUTION_PASS
